In [14]:
import os, json, re, time, getpass
import requests
import pandas as pd

# ==========================================
# STEP 1: Settings (Connect to the AI)
# ==========================================

In [15]:
HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "gemma4-26b-thinking"
TEMPERATURE = 0           # 0 = deterministic / predictable
SEED = 123

# Prompt for API Key
API_KEY = os.environ.get("UNI_API_KEY") or getpass.getpass("Paste the Uni API key: ")
print("Key loaded:", bool(API_KEY))

Key loaded: True


# ==========================================
# STEP 2: The Annotation Prompt
# ==========================================

In [16]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

# Customize your labels and definition here
annotation_prompt = """

ROLE
You are a text annotator specializing in media content analysis and computational communication. Your task is to apply the Stereotype Content Model (SCM) to evaluate whether targeted social groups in translated German newspaper articles are portrayed with Coldness (the opposite of Warmth in the SCM).

TASK
Based on the multiple paragraphs provided, decide whether [Gruppe 1] (or an individual member of [Gruppe 1]) is attributed with coldness. The label is TRUE if the text contains coldness and FALSE if it does not. Prioritize explicit mentions of cold attributes. Implicit portrayals are also encouraged to be coded when the text clearly conveys coldness without using a trait word. Give a short explanation of why you chose TRUE or FALSE, based on the criteria below. While coding, target only [Gruppe 1]. Do NOT code whether other groups ([andere Gruppe]) display coldness toward [Gruppe 1].

CONTEXT
You are working with paragraphs from German newspaper articles from a wide range of the political spectrum. The paragraphs have been translated into English. Each input consists of several consecutive paragraphs from one article, together with the target group [Gruppe 1]. Return one label for the whole passage. Entity names in the text are masked as [Gruppe 1] and [andere Gruppe]. Keep these names exactly as written and do not translate, change or replace them, including in your explanation.

DEFINITION
Coldness in news coverage is defined as an individual's or group's perceived negative intention towards the perceiver or their group of belonging. Coldness is conceptualized through two dimensions in media reporting:

Unfriendliness/asociality: Framing the individual or group as unfriendly if they are depicted as someone who treats others in an unfriendly, rude and disrespectful manner.
Attribute markers include but are not limited to: unfriendly, cold, unsociable, aloof, unpleasant, hostile, rude, disliked

Immorality: Framing the individual or group as immoral if they are attributed as someone untrustworthy, dishonest, unfair and bad.
Attribute markers include but are not limited to: immoral, untrustworthy, dishonest, insincere, unfair, intolerant, selfish, malicious

CRITERIA: COLDNESS OCCURS IF

Case: A cold trait is attributed to [Gruppe 1] or a member.
Example: "The [Gruppe 1] are considered unpleasant / rude."

Case: A synonym or noun form of a cold trait is used.
Example: "Their dishonesty is shameful."

Case: A behaviour that clearly shows bad intentions toward others.
Example: "The [Gruppe 1] deliberately ruined their neighbor's business."

Case: A negative moral evaluation.
Example: "The [Gruppe 1] selfishly hoarded resources."

CRITERIA: COLDNESS DOES NOT OCCUR IF

Case: Others are cold TOWARD the group.
Example: "The local population cheated [Gruppe 1] out of their share." / "The public showed no willingness to assist [Gruppe 1]."
Why not: The coldness belongs to the Germans, not to [Gruppe 1].

Case: The group is only a victim.
Example: "Three [Gruppe 1] were attacked."
Why not: Being a victim is not a trait.

Case: It is about ability or courage (competence/incompetence).
Example: "The cowardly [Gruppe 1]", "lazy"
Why not: Courage, diligence and skill, including their antonyms, = competence/incompetence.

Case: The cold word describes someone else.
Example: "Filip is rude to everyone," says the [Gruppe 1].
Why not: [Gruppe 1] is only the speaker.

Case: It is only a threat frame.
Example: "[Gruppe 1] weakens the economy."
Why not: An economic disadvantage, not a trait (Frames dimension).

Case: No portrayal at all.
Example: "In 2022, 12,000 [Gruppe 1] applied for asylum."
Why not: Facts and statistics only.

DECISION RULES
tba.

OUTPUT FORMAT
Return only a valid JSON object, with no text before or after it. The object has exactly two keys, in this order:
"explanation": 1-2 sentences explaining the label, citing the specific words or behavior that led to it.
"label": the string "TRUE" or "FALSE".

Example of valid output:
{"explanation": "The text calls the [Gruppe 1] rude and hostile toward their neighbors.", "label": "TRUE"}

"""

instructions = annotation_prompt
reminder = None  # this is just an additonal vairable to nudge the llm to a specific direction
JSON_MODE = True

print("--- Instructions to annotate COLD Loaded ---")
print(instructions)

--- Instructions to annotate COLD Loaded ---


ROLE
You are a text annotator specializing in media content analysis and computational communication. Your task is to apply the Stereotype Content Model (SCM) to evaluate whether targeted social groups in translated German newspaper articles are portrayed with Coldness (the opposite of Warmth in the SCM).

TASK
Based on the multiple paragraphs provided, decide whether [Gruppe 1] (or an individual member of [Gruppe 1]) is attributed with coldness. The label is TRUE if the text contains coldness and FALSE if it does not. Prioritize explicit mentions of cold attributes. Implicit portrayals are also encouraged to be coded when the text clearly conveys coldness without using a trait word. Give a short explanation of why you chose TRUE or FALSE, based on the criteria below. While coding, target only [Gruppe 1]. Do NOT code whether other groups ([andere Gruppe]) display coldness toward [Gruppe 1].

CONTEXT
You are working with paragraphs from Ger

# ==========================================
# STEP 3: Paragraph Input Data
# ==========================================

In [17]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

# SPECIFY CORRECT NAME
json_filename = "TRANSLATED_PARAGRAPHS.json"

try:
    with open(json_filename, "r", encoding="utf-8") as f:
        sample_data = json.load(f)
    sample = pd.DataFrame(sample_data)
    print(f"Successfully loaded {len(sample)} paragraph(s) from '{json_filename}'.")
except FileNotFoundError:
    print(f"Error: Could not find '{json_filename}' in the current folder. Please make sure the file exists.")

Successfully loaded 5000 paragraph(s) from 'TRANSLATED_PARAGRAPHS.json'.


# ==========================================
# STEP 4: The LLM API Call Function
# ==========================================

In [18]:
def call_llm(text, instructions, reminder=None, json_mode=False, max_retries=3):
    user_msg = f"Text: '{text}'"
    if reminder:
        user_msg += f"\nReminder: {reminder}"
        
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": instructions},
            {"role": "user",   "content": user_msg},
        ],
        "temperature": TEMPERATURE,
        "seed": SEED,
        "reasoning_effort": "none",
    }
    
    if json_mode:
        body["response_format"] = {"type": "json_object"}
        
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(f"{HOST}/chat/completions",
                              headers={"Authorization": f"Bearer {API_KEY}"},
                              json=body, timeout=120)
            r.raise_for_status()
            j = r.json()
            return {
                "output": j["choices"][0]["message"]["content"].strip(),
                "tokens_prompt": j["usage"]["prompt_tokens"],
                "tokens_response": j["usage"]["completion_tokens"],
            }
        except Exception as e:
            print(f"Attempt {attempt} failed: {e}")
            time.sleep(2 * attempt)
            
    return {"output": None, "tokens_prompt": None, "tokens_response": None}

# ==========================================
# STEP 5: Run Annotation Loop 
# ==========================================

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

results = []

for idx, row in sample.iterrows():
    start = time.time()
    res = call_llm(row["translation"], instructions, reminder, json_mode=JSON_MODE)
    seconds = round(time.time() - start, 1)
    
    raw = res["output"]
    parsed_json = {}
    parsed_ok = False
    
    try:
        parsed_json = json.loads(raw)
        parsed_ok = True
    except Exception:
        parsed_ok = False

    # Store results back in DataFrame
    sample.loc[idx, "raw_output"] = raw
    sample.loc[idx, "json_ok"] = parsed_ok
    sample.loc[idx, "sentiment"] = parsed_json.get("sentiment")
    sample.loc[idx, "topic"] = parsed_json.get("topic")
    sample.loc[idx, "summary_key_event"] = parsed_json.get("summary_key_event")
    sample.loc[idx, "seconds"] = seconds

    print(f"Paragraph {row['row_id']} processed in {seconds}s | JSON parsed: {parsed_ok}")
    print("Annotations:", json.dumps(parsed_json, ensure_ascii=False, indent=2))
    print("-" * 50)

KeyError: 'id'

# ==========================================
# STEP 6: Quality Checks & Verification
# ==========================================

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

valid_labels = {"TRUE", "FALSE"}

for idx, row in sample.iterrows():
    p_id = row.get("id", idx + 1)
    lbl = row.get("label")
    exp = str(row.get("explanation") or "")
    raw_text = str(row.get("text", ""))

    print(f"\n--- Quality Check [ID {p_id}] ---")
    
    # 1. Check JSON parsing
    json_ok = row.get("json_ok", False)
    print("JSON Valid:         ", "OK" if json_ok else "CHECK raw_output!")

    # 2. Check label validity
    label_ok = lbl in valid_labels
    print("Valid Label (T/F):  ", f"OK ({lbl})" if label_ok else f"CHECK! ({lbl})")

    # 3. Check explanation presence
    exp_ok = len(exp.strip()) > 0
    print("Explanation Present:", "OK" if exp_ok else "CHECK! (empty explanation)")

    # 4. Check entity masking preservation ([Gruppe 1])
    # Warns if [Gruppe 1] existed in input but model didn't keep it in explanation
    if "[Gruppe 1]" in raw_text:
        mask_ok = "[Gruppe 1]" in exp
        print("[Gruppe 1] Preserved:", "OK" if mask_ok else "CHECK! ([Gruppe 1] missing from explanation)")

: 

: 

# ==========================================
# STEP 7: Save the Results
# ==========================================

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

# Create a clean structured list for the JSON export
json_export_data = []

for idx, row in sample.iterrows():
    entry = {
        "id": int(row.get("id", idx + 1)),
        "text": row.get("text"),
        "cold_annotation": {
            "label": row.get("label"),           # "TRUE" or "FALSE"
            "explanation": row.get("explanation") # 1-2 sentence explanation
        },
        "metadata": {
            "json_parsed_ok": bool(row.get("json_ok", False)),
            "tokens_prompt": row.get("tokens_prompt"),
            "tokens_response": row.get("tokens_response"),
            "seconds": row.get("seconds")
        }
    }
    json_export_data.append(entry)

# Save to JSON file
json_output_path = "cold_annotation_results.json"
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(json_export_data, f, ensure_ascii=False, indent=2)

# Save to Excel file
excel_output_path = "cold_annotation_results.xlsx"
sample.to_excel(excel_output_path, index=False)

print(f"Finished! Saved annotations to '{json_output_path}' and '{excel_output_path}'.")

: 

: 